# Text Formatting

In [1]:
import pandas as pd
import numpy as np
import re
import os
import glob
import json
import seaborn as sns
import matplotlib.pyplot as plt

In [24]:
import re
import json

def extract_inscriptions_final(txt_file, output_jsonl):
    results = []

    with open(txt_file, "r", encoding="utf-8") as f:
        text = f.read()

    # Split pages
    pages = re.split(r"---\s*Page\s+(\d+)\s*---", text, flags=re.IGNORECASE)

    for i in range(1, len(pages), 2):
        page_num = int(pages[i])
        page_text = pages[i+1]
        lines = [line.strip() for line in page_text.splitlines() if line.strip()]

        j = 0
        while j < len(lines):
            line = lines[j]

            # Detect description (DOR)
            desc_match = re.match(r"^(\d+\..*?(?:տող\.|միատող\.))", line)
            if desc_match:
                entry = {
                    "page": page_num,
                    "description": desc_match.group(1).strip(),
                    "inscription_text": "",
                    "publication": None,
                    "note": None
                }

                # Collect inscription lines (INSCRIPTION)
                k = j + 1
                inscription_lines = []

                while k < len(lines):
                    nxt = lines[k]
                    if nxt.startswith("Հրատ."):
                        entry["publication"] = nxt
                        k += 1
                        break
                    elif nxt.startswith("Ծանոթ.") or re.match(r"^\d+\.", nxt):
                        break
                    elif not re.search(r"[ա-ֆ]", nxt):  # uppercase or OCR artifacts
                        inscription_lines.append(nxt)
                    k += 1

                entry["inscription_text"] = " ".join(inscription_lines).strip()

                # Collect note (if present)
                while k < len(lines):
                    nxt = lines[k]
                    if nxt.startswith("Ծանոթ."):
                        note_lines = [nxt]
                        m = k + 1
                        while m < len(lines):
                            nxt2 = lines[m]
                            if nxt2.startswith("Հրատ.") or nxt2.startswith("Ծանոթ.") or re.match(r"^\d+\.", nxt2):
                                break
                            note_lines.append(nxt2)
                            m += 1
                        entry["note"] = " ".join(note_lines).strip()
                        k = m
                        break
                    else:
                        k += 1

                results.append(entry)
                j = k
            else:
                j += 1

    # Write JSONL
    with open(output_jsonl, "w", encoding="utf-8") as out:
        for entry in results:
            out.write(json.dumps(entry, ensure_ascii=False) + "\n")
extract_inscriptions_final("C:/Users/Kamal/OneDrive/Desktop/PDS/corpus_txt/Divan_Prak10.txt", "divan10_harsh.jsonl")

# Kamal

In [197]:

def is_valid_title(line: str) -> bool:
    """
    Checks if a line is a valid inscription title.
    Format: number + dot, ALL CAPS TEXT (with punctuation allowed), optional rest.
    Example: '3. ԵՐԵՐՈՒՅԹԻ ՏԱԾԱՐ. բեմից հյուսիս, ...'
    """
    line = line.strip()
    # Allow uppercase Armenian + spaces + punctuation inside the title
    pattern = r"^\d+\.\s+[Ա-ՖԵՕՒՔ\s\.\-,:;()\[\]#]+?\."
    match = re.match(pattern, line)
    return bool(match)


In [198]:
def find_line(text,last_index,term):
    start_index = text.rfind("\n", 0, last_index)
    if start_index == -1:  # no newline found
        start_index = 0
    else:
        start_index += 1  # move to the character after '\n'

    # Extract the line
    line = text[start_index:last_index + len(term)]
    return line

In [199]:
import re

def extract_all_caps_block(text):

    snippet = text
    lines = snippet.splitlines()

    all_caps_blocks = []
    current_block = []

    # regex to match Armenian uppercase letters + digits + punctuation
    valid_pattern = re.compile(r"^[Ա-Ֆ0-9\s\.\-,:;#\[\]\(\)]+$")

    for line in lines:
        stripped = line.strip()
        if not stripped:
            if current_block:  # close block on empty line
                all_caps_blocks.append("\n".join(current_block))
                current_block = []
            continue

        # check if line is "all caps" or mostly uppercase
        if valid_pattern.match(stripped):
            current_block.append(stripped)
        else:
            if current_block:
                all_caps_blocks.append("\n".join(current_block))
                current_block = []

    if current_block:
        all_caps_blocks.append("\n".join(current_block))

    return all_caps_blocks


In [200]:
def generate_term_ranking(text,terms_of_interest=["Ծանոթ.", "տող.","միատող.","տողից."]):
    lists=[]
    for term in terms_of_interest:
        matches = [m.start() for m in re.finditer(term, text,re.IGNORECASE)]
        current_list=[]
        if any(term==desc_term for desc_term in ["միատող.","տող.","տողից."]): ## Check if the line containing a desribtion term is valid
            for index in matches:
                line = find_line(text,index,term)
                if is_valid_title(line):
                    current_list.append((term,index))
        else:
            current_list=[(term,m) for m in matches]

        lists.append(current_list)

    flattened = [item for sublist in lists for item in sublist]
    sorted_data = sorted(flattened, key=lambda x: x[1])
    return sorted_data


In [ ]:
with open("C:/Users/Kamal/OneDrive/Desktop/PDS/corpus_txt/Divan_Prak10.txt", "r", encoding="utf-8") as f:
    text = f.read()

pages = re.split(r"---\s*Page\s+(\d+)\s*---", text, flags=re.IGNORECASE)
corpus={}
for i in range(1, len(pages), 2):
        page_num = int(pages[i])
        page_text = pages[i+1]
        if any(keyword in page_text for keyword in ["Ծանոթ.", "տող.","Հրատ.","միատող."]):
              corpus[page_num]=page_text
              
items=list(corpus.items())
result=[]
for (key1, text1), (key2, text2) in zip(items, items[1:]):

    terms1=generate_term_ranking(text1)
    for (term1,idx1), (term2,idx2) in zip(terms1,terms1[1:]):
        if any(term1==desc_term for desc_term in ["միատող.","տող.","տողից."]) and term2=="Ծանոթ.":  ##case 1 describtion then note

            inscribtion_text=text1[idx1+len(term1):idx2-1]
            split=inscribtion_text.split("Հրատ.")
            publication=None

            if len(split)>1:
                publication="Հրատ. "+split[-1]
                inscribtion_text=split[0]

            note=text1[idx2:].split('\n\n')[0]

            title=find_line(text1,idx1,term1)

            current_entry={'page': key1,
                           'title':title.split('.')[0]+'.'+title.split('.')[1],
                            'inscribtion':inscribtion_text,
                            'publication':publication,
                            'note':note}
            
            result.append(current_entry)


        if any(term1==desc_term for desc_term in ["միատող.","տող.","տողից."]) and any(term2==desc_term for desc_term in ["միատող.","տող.","տողից."]):##Case 2 desc desc
            if term1=='միատող.' and term2=='տող.' and idx1-3==idx2:
                continue
            elif term2=='միատող.' and term1=='տող.' and idx1+3==idx2:
                continue
            else:
                title=find_line(text1,idx1,term1)
                inscribtion_text=text1[idx1+len(term1):idx2-1]

                split=inscribtion_text.split("Հրատ.")
                publication=None
                if len(split)>1:
                    publication="Հրատ. "+split[-1].split('\n')[0]
                    inscribtion_text=extract_all_caps_block(split[0])
                else:
                    inscribtion_text=extract_all_caps_block(inscribtion_text)
                
                current_entry={'page': key1,
                           'title':title.split('.')[0]+'.'+title.split('.')[1],
                            'inscribtion':" ".join(s for sub in inscribtion_text for s in sub),
                            'publication':publication,
                            'note':None}
                result.append(current_entry)
                    
    if key1+1==key2:
        terms2=generate_term_ranking(text2)
        if len(terms2)>0 and len(terms1)>0 :
            if any(terms1[-1][0]==desc_term for desc_term in ["միատող.","տող.","տողից."]) and terms2[0][0]=="Ծանոթ.":
                new_text=find_line(text1,terms1[-1][1],terms1[-1][0])+ text2
                new_terms=generate_term_ranking(new_text)

                idx1=new_terms[0][1]
                idx2=new_terms[1][1]

                term1=new_terms[0][0]
                term2=new_terms[1][0]


                inscribtion_text=text1[idx1+len(term1):idx2-1]
                split=inscribtion_text.split("Հրատ.")
                publication=None

                if len(split)>1:
                    publication="Հրատ. "+split[-1]
                    inscribtion_text=split[0]

                note=text1[idx2:].split('\n\n')[0]

                title=find_line(new_text,idx1,term1)
                current_entry={'page': key1,
                           'title':title.split('.')[0]+'.'+title.split('.')[1],
                            'inscribtion':inscribtion_text,
                            'publication':publication,
                            'note':note}
                result.append(current_entry)




        


In [209]:
len(result)

93

In [78]:

generate_term_ranking(corpus[22])


[('Ծանոթ.', 477),
 ('տող.', 497),
 ('տող.', 804),
 ('Ծանոթ.', 946),
 ('տող.', 1063),
 ('Ծանոթ.', 1261)]

In [72]:
text=corpus[22]
term="Ծանոթ."
matches = [m.start() for m in re.finditer(term, text)]
matches

[477, 946, 1261]

[('Հրատ.', 377),
 ('Ծանոթ.', 477),
 ('տող.', 497),
 ('Հրատ.', 523),
 ('տող.', 804),
 ('Ծանոթ.', 946),
 ('տող.', 1063),
 ('Հրատ.', 1225),
 ('Ծանոթ.', 1261)]

## To Do tomorrow
- Smooth text:
    - treat split text line returns (hell-o)
    - treat paragraph separation

## Merged POV